# Sanity Check - Step 04: Interpolate Bad Channels

Überprüft:
- Interpolation durchgeführt
- Bads-Liste geleert nach Interpolation
- Kanal-Anzahl gleich geblieben
- Amplituden noch plausibel

In [ ]:
import os
import sys
from pathlib import Path
import mne

root = Path.cwd()
candidate_roots = [root, root.parent, root.parent.parent]
for candidate in candidate_roots:
    pipeline_dir = candidate / "eeg_pipeline"
    if pipeline_dir.exists() and str(pipeline_dir) not in sys.path:
        sys.path.insert(0, str(pipeline_dir))
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

try:
    from eeg_pipeline import config
except ModuleNotFoundError:
    import config

print("Setup erfolgreich")

Setup erfolgreich


In [ ]:
# Manuelle Auswahl fuer diesen Notebook-Run
subject_id = "01"  # z.B. "02"
persons = ["P1", "P2"]  # oder nur ["P1"]
invalid = [p for p in persons if p not in {"P1", "P2"}]
if invalid:
    raise ValueError(f"Ungueltige Person(en): {invalid}. Erlaubt: P1,P2")
os.environ["EEG_SUBJECT"] = subject_id
os.environ["EEG_PERSONS"] = ",".join(persons)
print(f"Manuell gesetzt: EEG_SUBJECT={os.environ['EEG_SUBJECT']}, EEG_PERSONS={os.environ['EEG_PERSONS']}")

## 1. Before & After laden

In [ ]:
subject_id = os.getenv("EEG_SUBJECT", config.SUBJECTS[0]).strip()
persons_env = os.getenv("EEG_PERSONS", "")
if persons_env.strip():
    persons = [p.strip().upper() for p in persons_env.split(",") if p.strip()]
else:
    persons = ["P1", "P2"]

valid_persons = {"P1", "P2"}
invalid = [p for p in persons if p not in valid_persons]
if invalid:
    raise ValueError(f"Ungültige Person(en): {invalid}. Erlaubt: P1,P2")

for person in persons:
    before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
    
    if not before_path.exists():
        print(f"✗ {person}: Before-file not found")
        continue
    if not after_path.exists():
        print(f"✗ {person}: After-file not found")
        continue
    
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    
    print(f"✓ {person}: Both files loaded")

Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif...
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_interpolated.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\4026980368.py:14: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\4026980368.py:15: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_interpolated.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_after = mne.io.read_raw_fif(str(after_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
✓ P1: Both files loaded
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif...
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_interpolated.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\4026980368.py:14: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\4026980368.py:15: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_interpolated.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_after = mne.io.read_raw_fif(str(after_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
✓ P2: Both files loaded


## 2. Bad Channels Vergleich

In [ ]:
for person in persons:
    before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
    
    if not before_path.exists() or not after_path.exists():
        continue
    
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    
    bads_before = raw_before.info.get('bads', [])
    bads_after = raw_after.info.get('bads', [])
    
    print(f"\n=== {person} ===")
    print(f"Bad channels BEFORE interpolation: {len(bads_before)}")
    if bads_before:
        print(f"  {', '.join(bads_before)}")
    
    print(f"Bad channels AFTER interpolation: {len(bads_after)}")
    if bads_after:
        print(f"  {', '.join(bads_after)}")
    else:
        print(f"  ✓ Bad-Channels gelöscht nach Interpolation")

Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif...
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_interpolated.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\2141441670.py:8: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\2141441670.py:9: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_interpolated.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_after = mne.io.read_raw_fif(str(after_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.

=== P1 ===
Bad channels BEFORE interpolation: 0
Bad channels AFTER interpolation: 0
  ✓ Bad-Channels gelöscht nach Interpolation
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif...
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_interpolated.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\2141441670.py:8: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\2141441670.py:9: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_interpolated.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_after = mne.io.read_raw_fif(str(after_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.

=== P2 ===
Bad channels BEFORE interpolation: 0
Bad channels AFTER interpolation: 0
  ✓ Bad-Channels gelöscht nach Interpolation


## 3. Metadata Überprüfung

In [ ]:
for person in persons:
    before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_badchannels_detected.fif"
    after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
    
    if not before_path.exists() or not after_path.exists():
        continue
    
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    
    print(f"\n{person}:")
    
    # Channel count
    if len(raw_before.ch_names) == len(raw_after.ch_names):
        print(f"  ✓ Kanal-Anzahl erhalten: {len(raw_after.ch_names)}")
    else:
        print(f"  ✗ Kanal-Anzahl geändert: {len(raw_before.ch_names)} -> {len(raw_after.ch_names)}")
    
    # Sampling rate
    if raw_before.info['sfreq'] == raw_after.info['sfreq']:
        print(f"  ✓ Sampling rate gleich: {raw_after.info['sfreq']} Hz")
    else:
        print(f"  ✗ Sampling rate geändert: {raw_before.info['sfreq']} -> {raw_after.info['sfreq']}")
    
    # Sample count
    if raw_before.n_times == raw_after.n_times:
        print(f"  ✓ Sample-Anzahl gleich: {raw_after.n_times}")
    else:
        print(f"  ✗ Sample-Anzahl geändert: {raw_before.n_times} -> {raw_after.n_times}")

Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif...
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_interpolated.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\3124674865.py:8: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\3124674865.py:9: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_interpolated.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_after = mne.io.read_raw_fif(str(after_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.

P1:
  ✓ Kanal-Anzahl erhalten: 72
  ✓ Sampling rate gleich: 200.0 Hz
  ✓ Sample-Anzahl gleich: 733800
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif...
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_interpolated.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\3124674865.py:8: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_badchannels_detected.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
C:\Users\BKALYON\AppData\Local\Temp\ipykernel_35424\3124674865.py:9: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_interpolated.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_after = mne.io.read_raw_fif(str(after_path), preload=False)


    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.

P2:
  ✓ Kanal-Anzahl erhalten: 72
  ✓ Sampling rate gleich: 200.0 Hz
  ✓ Sample-Anzahl gleich: 733800
